# GMF candidate-generation baseline

A static implicit-feedback generalised matrix factorisation (GMF) baseline built
from the cached binary user-item matrix.

- Reuses `artifacts/cf/user_item_train.npz`; no new interaction matrix is built.
- The model contains exactly two `nn.Embedding` blocks: one for users and one for
  articles. The score is their raw dot product. There is no MLP, bias or side
  feature.
- Training uses BPR with one uniformly sampled negative per positive.
- Validation candidates exclude the request's full causal history and use the
  eligible catalogue at the request timestamp.

The notebook runs the full pipeline by default. Set `GMF_FULL_RUN=0` to execute a
small smoke run.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))

from cf_models import build_interaction_matrix, load_context, resolve_device

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FULL_RUN = os.environ.get("GMF_FULL_RUN", "1") == "1"
RUN_TAG = "gmf" if FULL_RUN else "gmf_smoke"
MODEL_NAME = "gmf"
ARTIFACT_DIR = ARTIFACTS / RUN_TAG
RECOMMENDATIONS_PATH = ARTIFACT_DIR / f"recommendations_{TARGET_SPLIT}_top500.parquet"
RESULT_PATH = RESULTS / f"candidate_generation_{MODEL_NAME}_{TARGET_SPLIT}.csv"
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
EMBED_DIM = 128
NUM_EPOCHS = 20 if FULL_RUN else 2
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SEED = 4222
SMOKE_PAIRS = 300_000
SMOKE_REQUESTS = 3_000
DEVICE = resolve_device("auto")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Mode:", "full" if FULL_RUN else "smoke")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

Repository: c:\Users\Yong Rui\BT4222-Group-6-
Mode: full
Device: cuda
Torch: 2.13.0a0+rocm7.13.0a20260416


## 1. Reuse the cached user-item matrix

The matrix is loaded from `artifacts/cf/user_item_train.npz`. Its row index is
`user_code` and its column index is `article_row`; column `0` is padding.

In [2]:
context = load_context(ROOT)
matrix, matrix_manifest = build_interaction_matrix(ROOT, context, rebuild=False)
matrix = matrix.tocsr()
expected_shape = (context.n_users, context.n_items)
if matrix.shape != expected_shape:
    raise ValueError(f"Matrix shape {matrix.shape} != {expected_shape}")
if matrix.nnz == 0:
    raise ValueError("Cached matrix has no interactions")

all_users = np.repeat(
    np.arange(matrix.shape[0], dtype=np.int64), np.diff(matrix.indptr)
)
all_items = matrix.indices.astype(np.int64, copy=False)
if all_items.min() < 1 or all_items.max() >= context.n_items:
    raise ValueError("Cached matrix contains an invalid article row")

if FULL_RUN:
    train_users, train_items = all_users, all_items
else:
    rng = np.random.default_rng(SEED)
    if all_items.size > SMOKE_PAIRS:
        keep = rng.choice(all_items.size, SMOKE_PAIRS, replace=False)
        train_users = all_users[keep]
        train_items = all_items[keep]
    else:
        train_users, train_items = all_users, all_items

print(f"Users: {context.n_users:,}")
print(f"Articles: {context.n_items:,}")
print(f"Interactions: {matrix.nnz:,}")
print(f"Training pairs: {len(train_users):,}")
print("Cutoff:", matrix_manifest["cutoff"])

Users: 750,434
Articles: 104,152
Interactions: 15,552,485
Training pairs: 15,552,485
Cutoff: 2019-11-14 07:30:28


## 2. Two embeddings and a dot product

`GMF` has exactly two embedding tables. BPR compares the dot product of a
positive pair with that of one uniformly sampled negative article. No
false-negative filtering is applied.

In [3]:
class GMF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

    def forward(self, user_ids, item_ids):
        return (
            self.user_embedding(user_ids) * self.item_embedding(item_ids)
        ).sum(dim=-1)


def train_epoch(model, optimizer, user_ids, item_ids, rng):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(
            user_ids[batch], dtype=torch.long, device=DEVICE
        )
        positives = torch.as_tensor(
            item_ids[batch], dtype=torch.long, device=DEVICE
        )
        negatives = torch.randint(
            1,
            context.n_items,
            (len(batch),),
            dtype=torch.long,
            device=DEVICE,
        )
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(users, negatives)
        loss = -F.logsigmoid(positive_scores - negative_scores).mean()
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


model = GMF(context.n_users, context.n_items, EMBED_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
rng = np.random.default_rng(SEED + 1)
training_log = []
for epoch in range(1, NUM_EPOCHS + 1):
    started = time.time()
    loss = train_epoch(model, optimizer, train_users, train_items, rng)
    elapsed = time.time() - started
    training_log.append(
        {"epoch": epoch, "bpr_loss": loss, "seconds": elapsed}
    )
    print(f"epoch {epoch:02d} bpr_loss={loss:.6f} ({elapsed:.1f}s)")

torch.save(
    {
        "state_dict": model.state_dict(),
        "config": {
            "n_users": context.n_users,
            "n_items": context.n_items,
            "embed_dim": EMBED_DIM,
            "objective": "bpr",
            "num_negatives": 1,
            "epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "matrix_manifest": matrix_manifest,
        },
    },
    ARTIFACT_DIR / "model.pt",
)
pl.DataFrame(training_log).write_csv(ARTIFACT_DIR / "training_log.csv")
print("Saved model:", ARTIFACT_DIR / "model.pt")

epoch 01 bpr_loss=0.409035 (12.2s)
epoch 02 bpr_loss=0.099362 (10.9s)
epoch 03 bpr_loss=0.087313 (10.9s)
epoch 04 bpr_loss=0.083739 (10.9s)
epoch 05 bpr_loss=0.081286 (10.9s)
epoch 06 bpr_loss=0.079271 (10.9s)
epoch 07 bpr_loss=0.076893 (10.9s)
epoch 08 bpr_loss=0.074539 (10.9s)
epoch 09 bpr_loss=0.072211 (10.9s)
epoch 10 bpr_loss=0.069208 (10.9s)
epoch 11 bpr_loss=0.066318 (10.9s)
epoch 12 bpr_loss=0.063402 (10.9s)
epoch 13 bpr_loss=0.060570 (10.9s)
epoch 14 bpr_loss=0.057601 (10.9s)
epoch 15 bpr_loss=0.054894 (10.9s)
epoch 16 bpr_loss=0.052217 (10.9s)
epoch 17 bpr_loss=0.049643 (10.9s)
epoch 18 bpr_loss=0.047167 (10.9s)
epoch 19 bpr_loss=0.045002 (10.9s)
epoch 20 bpr_loss=0.042904 (10.9s)
Saved model: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\gmf\model.pt


## 3. Candidate helpers

These helpers keep the candidate format, eligibility policy, history exclusion
and popularity tie-breaking consistent with the other candidate generators.

In [4]:
CANDIDATE_SCHEMA = pa.schema(
    [
        ("impression_uid", pa.int64()),
        ("cutoff", pa.int16()),
        ("news_code", pa.int32()),
    ]
)


def popularity_select(order, n_eligible, excluded=None, k=CANDIDATE_CAP):
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(
        f"only {len(selected)} popular eligible articles; cannot select {k}"
    )


def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    if pool.size < count:
        raise ValueError(
            f"only {pool.size} eligible articles remain; cannot fill {count}"
        )
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    if scores.ndim != 1:
        raise ValueError("scores must be one-dimensional")
    if popularity.shape != scores.shape or news_code.shape != scores.shape:
        raise ValueError("score, popularity and news_code shapes must match")
    if k < 1:
        raise ValueError("k must be positive")
    n = scores.size
    if n < k:
        raise ValueError(f"only {n} eligible articles; cannot select {k}")
    if fallback_order is not None:
        return popularity_select(
            fallback_order, n, excluded=excluded, k=k
        )

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        if positions.min() < 0 or positions.max() >= n:
            raise ValueError("excluded position outside the eligible array")
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(
            available, popularity, news_code, k
        )

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort(
                (news_code[ties], -popularity[ties])
            )
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = popularity_fill(
            remaining, popularity, news_code, k - selected.size
        )
        selected = np.r_[selected, fill]

    return selected


class CandidateWriter:
    def __init__(self, path, batch_rows=2_000_000):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.batch_rows = batch_rows
        self.frames = []
        self.pending = 0
        self.rows = 0
        self.writer = pq.ParquetWriter(
            self.path, CANDIDATE_SCHEMA, compression="zstd"
        )

    def add(self, impression_uid, cutoff, news_code):
        frame = pa.table(
            {
                "impression_uid": pa.array(
                    impression_uid, type=pa.int64()
                ),
                "cutoff": pa.array(cutoff, type=pa.int16()),
                "news_code": pa.array(news_code, type=pa.int32()),
            },
            schema=CANDIDATE_SCHEMA,
        )
        self.frames.append(frame)
        self.pending += frame.num_rows
        self.rows += frame.num_rows
        if self.pending >= self.batch_rows:
            self.flush()

    def flush(self):
        if not self.frames:
            return
        table = pa.concat_tables(self.frames)
        self.writer.write_table(table)
        self.frames = []
        self.pending = 0

    def close(self):
        self.flush()
        self.writer.close()

## 4. Score the eligible catalogue and write Top-500 candidates

Each request uses the static embedding of its `user_code`. Articles not yet
available at the request timestamp are removed, every article in the request's
`full_history_rows` is excluded, and empty-history requests fall back to
popularity order. Ties are broken by popularity and then `news_code`.

In [5]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No request profiles for {split}")
    frame = pl.read_parquet(
        parts,
        columns=[
            "impression_uid",
            "user_code",
            "timestamp",
            "full_history_rows",
        ],
    )
    if frame["impression_uid"].n_unique() != frame.height:
        raise ValueError("Duplicate impression_uid in request profiles")
    if frame["full_history_rows"].null_count():
        raise ValueError("Null full_history_rows in request profiles")
    return frame.sort("timestamp", maintain_order=True)


catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, context.n_items, dtype=np.int32),
        "news_code": context.news_code[1:],
        "available_from": context.available_from[1:],
        "popularity": context.popularity[1:],
    }
)
ordered = catalogue.sort(["available_from", "news_code"])
article_rows = ordered["article_row"].to_numpy().astype(np.int64)
news_sorted = ordered["news_code"].to_numpy().astype(np.int64)
available_sorted = ordered["available_from"].to_numpy().astype(
    "datetime64[ns]"
)
popularity_sorted = ordered["popularity"].to_numpy().astype(np.float32)
pos_in_sorted = np.full(context.n_items, -1, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(article_rows.size, dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))
if not np.all(available_sorted[1:] >= available_sorted[:-1]):
    raise ValueError("Availability order is not non-decreasing")

profiles = load_profiles(TARGET_SPLIT)
impression_uids = profiles["impression_uid"].to_numpy()
target_users = profiles["user_code"].to_numpy().astype(np.int64)
timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
history_rows = profiles["full_history_rows"].to_list()
if target_users.min() < 0 or target_users.max() >= context.n_users:
    raise ValueError("Profile user_code is outside the matrix")
has_history = np.fromiter(
    (len(rows) > 0 for rows in history_rows),
    dtype=bool,
    count=len(history_rows),
)

starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts, ends))
if not FULL_RUN:
    selected_batches = []
    total = 0
    for start, end in batches:
        selected_batches.append((start, end))
        total += end - start
        if total >= SMOKE_REQUESTS:
            break
    batches = selected_batches
processed_requests = int(sum(end - start for start, end in batches))
print(f"Scoring requests: {processed_requests:,} / {len(timestamps):,}")

user_emb = model.user_embedding.weight.detach().cpu().numpy()[target_users]
item_emb = model.item_embedding.weight.detach().cpu().numpy()[article_rows]
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
writer = CandidateWriter(RECOMMENDATIONS_PATH)
started = time.time()
try:
    for batch_no, (start, end) in enumerate(batches, 1):
        timestamp = timestamps[start]
        n_eligible = int(
            np.searchsorted(available_sorted, timestamp, side="right")
        )
        if n_eligible < CANDIDATE_CAP:
            raise ValueError(
                f"Only {n_eligible} eligible articles at {timestamp}"
            )
        scores = user_emb[start:end] @ item_emb[:n_eligible].T
        popularity_batch = popularity_sorted[:n_eligible]
        news_batch = news_sorted[:n_eligible]
        selected_batch = np.empty(
            (end - start, CANDIDATE_CAP), dtype=np.int64
        )
        for local, profile_row in enumerate(range(start, end)):
            rows = history_rows[profile_row]
            if rows:
                excluded = pos_in_sorted[
                    np.asarray(rows, dtype=np.int64)
                ]
                excluded = excluded[
                    (excluded >= 0) & (excluded < n_eligible)
                ]
            else:
                excluded = np.empty(0, dtype=np.int64)
            fallback = (
                None
                if has_history[profile_row]
                else global_popularity_order
            )
            selected_batch[local] = select_top_k(
                scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=fallback,
            )

        writer.add(
            np.repeat(impression_uids[start:end], CANDIDATE_CAP),
            np.tile(cutoff_values, end - start),
            news_batch[selected_batch.reshape(-1)],
        )
        if batch_no % 1000 == 0 or batch_no == len(batches):
            print(
                f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                f"{writer.rows:,} rows",
                flush=True,
            )
finally:
    writer.close()

elapsed = time.time() - started
expected_rows = processed_requests * CANDIDATE_CAP
actual_rows = (
    pl.scan_parquet(RECOMMENDATIONS_PATH)
    .select(pl.len())
    .collect()
    .item()
)
if actual_rows != expected_rows:
    raise ValueError(
        f"Expected {expected_rows:,} rows, found {actual_rows:,}"
    )
if not FULL_RUN:
    check = pl.read_parquet(RECOMMENDATIONS_PATH)
    if check.select(pl.col("cutoff").n_unique()).item() != CANDIDATE_CAP:
        raise ValueError("Smoke output is missing cutoff values")
    if check.group_by("impression_uid").agg(
        pl.col("news_code").n_unique().alias("unique_news")
    ).filter(pl.col("unique_news") != CANDIDATE_CAP).height:
        raise ValueError("Smoke output contains duplicate candidates")
print(f"Wrote {actual_rows:,} candidate rows in {elapsed:.1f}s")
print("Candidates:", RECOMMENDATIONS_PATH)

Scoring requests: 334,913 / 334,913
Scored 1,000/54,728 timestamp batches; 4,416,000 rows
Scored 2,000/54,728 timestamp batches; 9,115,500 rows
Scored 3,000/54,728 timestamp batches; 13,939,000 rows
Scored 4,000/54,728 timestamp batches; 18,893,500 rows
Scored 5,000/54,728 timestamp batches; 23,876,000 rows
Scored 6,000/54,728 timestamp batches; 28,971,500 rows
Scored 7,000/54,728 timestamp batches; 34,127,500 rows
Scored 8,000/54,728 timestamp batches; 39,024,500 rows
Scored 9,000/54,728 timestamp batches; 43,858,500 rows
Scored 10,000/54,728 timestamp batches; 48,477,500 rows
Scored 11,000/54,728 timestamp batches; 53,004,500 rows
Scored 12,000/54,728 timestamp batches; 57,627,000 rows
Scored 13,000/54,728 timestamp batches; 62,315,000 rows
Scored 14,000/54,728 timestamp batches; 67,170,000 rows
Scored 15,000/54,728 timestamp batches; 71,848,500 rows
Scored 16,000/54,728 timestamp batches; 77,027,000 rows
Scored 17,000/54,728 timestamp batches; 81,893,500 rows
Scored 18,000/54,728 ti

## 5. Manifest, evaluation and comparison

Full runs use the shared `CandidateEvaluator`. Smoke runs only check the
candidate table structurally because they intentionally score a subset.

In [6]:
manifest = {
    "model": MODEL_NAME,
    "score": "raw dot product of user and article embeddings",
    "embedding_blocks": ["user_embedding", "item_embedding"],
    "objective": "BPR with one uniformly sampled negative per positive",
    "matrix": "artifacts/cf/user_item_train.npz",
    "matrix_cutoff": matrix_manifest["cutoff"],
    "fit_split": "train",
    "target_split": TARGET_SPLIT,
    "candidate_cap": CANDIDATE_CAP,
    "candidate_format": ["impression_uid", "cutoff", "news_code"],
    "eligibility": "available_from <= timestamp",
    "history_policy": "exclude every article in full_history_rows",
    "fallback": "popularity order for empty-history requests or short lists",
    "mode": "full" if FULL_RUN else "smoke",
    "seed": SEED,
    "device": DEVICE,
    "torch_version": torch.__version__,
    "embed_dim": EMBED_DIM,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "train_pairs": int(len(train_users)),
    "matrix_nonzero": int(matrix.nnz),
    "target_requests": int(processed_requests),
    "recommendation_rows": int(actual_rows),
    "recommendations_file": RECOMMENDATIONS_PATH.name,
    "model_file": "model.pt",
    "created_at": datetime.now(timezone.utc).isoformat(),
}
manifest_path = ARTIFACT_DIR / f"manifest_{TARGET_SPLIT}.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Manifest:", manifest_path)

if FULL_RUN:
    from cg_evaluation import CandidateEvaluator

    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate(
        RECOMMENDATIONS_PATH,
        model_name=MODEL_NAME,
        m_values=M_VALUES,
    )
    evaluation.write_csv(RESULT_PATH)

    if COMPARISON_PATH.exists():
        existing = pl.read_csv(COMPARISON_PATH).filter(
            pl.col("model") != MODEL_NAME
        )
        common = [
            column
            for column in existing.columns
            if column in evaluation.columns
        ]
        comparison = pl.concat(
            [existing.select(common), evaluation.select(common)],
            how="vertical_relaxed",
        )
    else:
        comparison = evaluation
    comparison.write_csv(COMPARISON_PATH)
    print(evaluation.sort("m"))
    print("Results:", RESULT_PATH)
else:
    print("Smoke run: shared evaluator skipped; structural checks passed.")

Manifest: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\gmf\manifest_validation.json
shape: (4, 12)
┌───────┬────────────┬─────┬─────────────┬───┬─────────────┬─────────────┬────────────┬────────────┐
│ model ┆ split      ┆ m   ┆ recall_pool ┆ … ┆ retrieved_r ┆ mean_candid ┆ mean_eligi ┆ eligible_c │
│ ---   ┆ ---        ┆ --- ┆ ed          ┆   ┆ elevant     ┆ ates        ┆ ble_pool   ┆ lick_cover │
│ str   ┆ str        ┆ i64 ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---        ┆ age        │
│       ┆            ┆     ┆ f64         ┆   ┆ i64         ┆ f64         ┆ f64        ┆ ---        │
│       ┆            ┆     ┆             ┆   ┆             ┆             ┆            ┆ f64        │
╞═══════╪════════════╪═════╪═════════════╪═══╪═════════════╪═════════════╪════════════╪════════════╡
│ gmf   ┆ validation ┆ 50  ┆ 0.008915    ┆ … ┆ 4588        ┆ 50.0        ┆ 99816.1906 ┆ 1.0        │
│       ┆            ┆     ┆             ┆   ┆             

## 6. Interpretation and limitations

- The user representation is a static `user_code` embedding, not a causal
  request profile. Full `full_history_rows` is used only for exclusion and
  fallback.
- The matrix is built from all training interactions; training has no temporal
  holdout or early stopping.
- Negatives are sampled uniformly without filtering false negatives.
- The dot product is a ranking score, not a calibrated click probability.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.